# ETHICS Gender Persona — Overall Linguistic Markers v1.0

Notebook ini adalah tampilan utama **overall ETHICS**. Empat subset
(`commonsense`, `deontology`, `justice`, `virtue`) digabung hanya untuk
analisis linguistic marker dan response behavior. Detail per-subset tetap
dipertahankan sebagai lampiran pada notebook/bundle v1.4 r4.

Tidak dibuat satu angka accuracy ETHICS gabungan karena arti label dan
konstruk empat subset berbeda. Semua hasil masih berstatus `candidate`.
Analisis utama memakai visible text yang tersedia, termasuk `abstain`;
sensitivity memakai pasangan yang kedua responsnya `committed`.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def locate_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'src').is_dir() and (candidate / 'result').is_dir():
            return candidate
    raise FileNotFoundError('Repo root D:/TugasAkhir tidak ditemukan.')


REPO_ROOT = locate_repo_root()
OVERALL_DIR = (
    REPO_ROOT
    / 'result/benchmarks/ethics/reports/Gender_Persona/linguistic_markers/analysis'
    / 'gender_ethics_linguistic_markers_overall_candidate_r2_20260905'
)
APPENDIX_DIR = (
    REPO_ROOT
    / 'result/benchmarks/ethics/reports/Gender_Persona/linguistic_markers/analysis'
    / 'gender_ethics_linguistic_markers_v1_4_candidate_r4_20260826'
)
assert OVERALL_DIR.is_dir(), 'Jalankan builder overall terlebih dahulu.'
manifest = json.loads((OVERALL_DIR / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['result_status'] == 'candidate'
assert manifest['accuracy_aggregated_across_subsets'] is False
assert manifest['total_responses'] == 81_756
assert manifest['visible_text_records'] == 80_840

MODEL_LABELS = {
    'meta-llama_llama-3.1-8b-instruct': 'Llama 3.1 8B',
    'meta-llama_llama-3.3-70b-instruct': 'Llama 3.3 70B',
    'mistralai_ministral-8b-2512': 'Ministral 8B',
    'deepseek_deepseek-r1-distill-qwen-32b': 'DeepSeek R1 32B',
    'openai_gpt-oss-20b': 'GPT-OSS 20B',
    'qwen_qwen3-next-80b-a3b-thinking': 'Qwen3 Next 80B',
}
PRIMARY_MARKERS = manifest['primary_marker_outcomes']
SUBSETS = manifest['subsets']

print(f"Bundle       : {manifest['bundle_version']}")
print(f"Scope        : {manifest['analysis_scope']}")
print(f"Total records: {manifest['total_responses']:,}")
print(f"Visible text : {manifest['visible_text_records']:,}")
print(f"Subsets      : {', '.join(SUBSETS)}")
print(f"Output       : {OVERALL_DIR}")

## 1. Overall response status

Tabel utama diringkas lintas empat subset, tetapi `total`, `committed`,
`abstain`, `missing`, dan `text_available` tetap ditampilkan agar coverage
tidak tersembunyi.

In [ ]:
status = pd.read_csv(OVERALL_DIR / 'overall_response_status.csv')
persona_status = (
    status.groupby('persona', as_index=False)[
        ['total', 'committed', 'abstain', 'missing', 'text_available', 'technical_only']
    ].sum()
)
persona_status['commit_rate'] = persona_status['committed'] / persona_status['total']
persona_status['abstain_rate'] = persona_status['abstain'] / persona_status['total']
persona_status['missing_rate'] = persona_status['missing'] / persona_status['total']
persona_status['text_available_rate'] = persona_status['text_available'] / persona_status['total']
display(persona_status.style.format({
    'commit_rate': '{:.2%}',
    'abstain_rate': '{:.2%}',
    'missing_rate': '{:.2%}',
    'text_available_rate': '{:.2%}',
}))

display(Markdown(
    '**Catatan:** perbedaan response status antar-persona adalah hasil yang relevan dan bukan alasan untuk menghapus `abstain` dari analisis utama.'
))

## 2. Overall marker profile

Profil berikut menggabungkan empat subset untuk setiap persona dan seluruh
model. `mean_density_per_100_words` digabung secara berbobot berdasarkan
jumlah respons linguistically available. Marker tetap dilaporkan sebagai
surface/lexical proxy, bukan bukti langsung reasoning moral.

In [ ]:
marker_profile = pd.read_csv(OVERALL_DIR / 'overall_marker_profile_41.csv')
primary_profile = marker_profile[marker_profile['marker_id'].isin(PRIMARY_MARKERS)].copy()
primary_profile['density_sum'] = (
    primary_profile['mean_density_per_100_words'] * primary_profile['n_available']
)
overall_marker = (
    primary_profile.groupby(['persona', 'marker_id'], as_index=False)
    .agg(
        n_available=('n_available', 'sum'),
        responses_present=('responses_present', 'sum'),
        total_occurrences=('total_occurrences', 'sum'),
        density_sum=('density_sum', 'sum'),
    )
)
overall_marker['response_prevalence_pct'] = (
    overall_marker['responses_present'] / overall_marker['n_available'] * 100
)
overall_marker['mean_density_per_100_words'] = (
    overall_marker['density_sum'] / overall_marker['n_available']
)
overall_marker = overall_marker.drop(columns='density_sum')
display(
    overall_marker.sort_values(['marker_id', 'persona'])
    .style.format({
        'response_prevalence_pct': '{:.2f}',
        'mean_density_per_100_words': '{:.4f}',
    })
)

surface_profile = pd.read_csv(OVERALL_DIR / 'overall_surface_profile_3.csv')
display(Markdown('### Surface outcomes pooled across models'))
surface_by_persona = (
    surface_profile.groupby(['persona', 'outcome_id'], as_index=False)
    .agg(
        n_available=('n_available', 'sum'),
        mean=('mean', 'mean'),
        median_of_model_medians=('median', 'median'),
    )
)
display(surface_by_persona.style.format({
    'mean': '{:.3f}',
    'median_of_model_medians': '{:.3f}',
}))

## 3. Overall paired persona effects

Perbandingan di bawah memakai seluruh question ID dari empat subset.
`subset` tetap disimpan di source pair key, tetapi tidak ditampilkan sebagai
panel utama. Arah delta adalah persona kiri dikurangi persona kanan.
Semua p-value dan effect masih berstatus candidate.

In [ ]:
paired = pd.read_csv(OVERALL_DIR / 'overall_paired_numeric_21.csv')
all_text = paired[paired['pair_filter'].eq('all_visible_text')].copy()
all_text['abs_rank_biserial'] = all_text['rank_biserial'].abs()
top_overall = (
    all_text.sort_values('abs_rank_biserial', ascending=False)
    .loc[:, [
        'analysis_family', 'model', 'identity', 'contrast_id', 'outcome_id',
        'n_pairs', 'median_paired_delta', 'rank_biserial', 'ci_low',
        'ci_high', 'p_holm',
    ]]
    .head(50)
)
display(top_overall.style.format({
    'median_paired_delta': '{:+.4f}',
    'rank_biserial': '{:+.4f}',
    'ci_low': '{:+.4f}',
    'ci_high': '{:+.4f}',
    'p_holm': '{:.4g}',
}))

display(Markdown(
    '**Interpretasi:** tabel ini menunjukkan perbedaan surface marker pada seluruh ETHICS, bukan satu skor moral reasoning dan bukan aggregate accuracy.'
))

## 4. Sensitivity: all visible versus both committed

In [ ]:
key = ['analysis_family', 'model_id', 'identity', 'outcome_id', 'contrast_id']
all_effects = paired[paired['pair_filter'].eq('all_visible_text')][
    key + ['n_pairs', 'rank_biserial']
].rename(columns={'n_pairs': 'n_pairs_all', 'rank_biserial': 'rank_biserial_all'})
committed_effects = paired[paired['pair_filter'].eq('both_committed')][
    key + ['n_pairs', 'rank_biserial']
].rename(columns={'n_pairs': 'n_pairs_committed', 'rank_biserial': 'rank_biserial_committed'})
sensitivity = all_effects.merge(committed_effects, on=key, how='outer')
sensitivity['pair_retention'] = sensitivity['n_pairs_committed'] / sensitivity['n_pairs_all']
sensitivity['same_direction'] = (
    np.sign(sensitivity['rank_biserial_all']) == np.sign(sensitivity['rank_biserial_committed'])
)
sensitivity_summary = (
    sensitivity.groupby('analysis_family', as_index=False)
    .agg(
        comparisons=('outcome_id', 'size'),
        median_pair_retention=('pair_retention', 'median'),
        same_direction_rate=('same_direction', 'mean'),
    )
)
display(sensitivity_summary.style.format({
    'median_pair_retention': '{:.2%}',
    'same_direction_rate': '{:.2%}',
}))

## 5. Visualisasi overall ETHICS

Visualisasi ini mengikuti bentuk visual pada pipeline v1.4.1/r4: coverage
heatmap, paired-effect panels, sensitivity concordance, dan opening
composition. Perbedaannya hanya dimensi subset diolah sebagai
`overall_ethics`; visual per-subset tetap berada di lampiran.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

FIG_DIR = OVERALL_DIR / 'images'
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')

PERSONA_ORDER = [
    'neutral', 'man_explicit_1', 'man_implicit_1', 'man_implicit_2', 'man_implicit_3',
    'woman_explicit_1', 'woman_implicit_1', 'woman_implicit_2', 'woman_implicit_3',
]
MODEL_ORDER = list(MODEL_LABELS)
MODEL_COLORS = dict(zip(MODEL_ORDER, sns.color_palette('tab10', n_colors=len(MODEL_ORDER)).as_hex()))
OUTCOME_ORDER = manifest['primary_surface_outcomes'] + manifest['primary_marker_outcomes']
OUTCOME_LABELS = {item: item.replace('_', ' ') for item in OUTCOME_ORDER}
CONTRAST_LABELS = {
    'explicit_vs_neutral': 'E − N',
    'implicit_1_vs_neutral': 'I1 − N',
    'implicit_2_vs_neutral': 'I2 − N',
    'implicit_3_vs_neutral': 'I3 − N',
    'implicit_1_vs_explicit': 'I1 − E',
    'implicit_2_vs_explicit': 'I2 − E',
    'implicit_3_vs_explicit': 'I3 − E',
    'man_minus_woman_explicit': 'E: Man − Woman',
    'man_minus_woman_implicit_1': 'I1: Man − Woman',
    'man_minus_woman_implicit_2': 'I2: Man − Woman',
    'man_minus_woman_implicit_3': 'I3: Man − Woman',
}

figure_rows = []

# 5.1 Same coverage heatmap as the active v1.4.1 figure, pooled over subsets.
coverage_matrix = (
    status.pivot(index='model', columns='persona', values='text_available_rate')
    .reindex(index=[MODEL_LABELS[item] for item in MODEL_ORDER], columns=PERSONA_ORDER)
    * 100
)
fig, ax = plt.subplots(figsize=(22, 8.5))
sns.heatmap(
    coverage_matrix, cmap='Blues', vmin=max(0, float(coverage_matrix.min().min()) - 2),
    vmax=100, annot=True, fmt='.1f', annot_kws={'fontsize': 12, 'fontweight': 'bold'},
    linewidths=0.8, linecolor='white', cbar_kws={'label': 'Visible text tersedia (%)'}, ax=ax,
)
ax.set_title('Overall ETHICS — cakupan visible text per model dan persona', fontweight='bold')
ax.set_xlabel('Persona')
ax.set_ylabel('Model')
ax.tick_params(axis='x', rotation=35)
fig.tight_layout()
fig.savefig(FIG_DIR / '01_overall_visible_text_coverage.png', dpi=180, bbox_inches='tight')
figure_rows.append(('01_overall_visible_text_coverage.png', 'Overall visible-text coverage by model and persona.'))
display(fig)
plt.close(fig)

def plot_overall_within_effects(table, contrast_ids, filename, title):
    data = table[(table['analysis_family'].eq('within_identity')) & table['contrast_id'].isin(contrast_ids)].copy()
    y_lookup = {outcome: len(OUTCOME_ORDER) - 1 - index for index, outcome in enumerate(OUTCOME_ORDER)}
    fig, axes = plt.subplots(1, len(contrast_ids), figsize=(6.0 * len(contrast_ids), 16.0), sharex=True, sharey=True)
    axes = np.atleast_1d(axes)
    for ax, contrast in zip(axes, contrast_ids):
        panel = data[data['contrast_id'].eq(contrast)]
        for _, point in panel.iterrows():
            if pd.isna(point['rank_biserial']):
                continue
            ax.scatter(
                point['rank_biserial'], y_lookup[point['outcome_id']], s=55,
                marker='o' if point['identity'] == 'man' else '^',
                color=MODEL_COLORS.get(point['model_id'], '#64748B'),
                alpha=0.95 if point['p_holm'] < 0.05 else 0.45,
                edgecolors='white', linewidths=0.6,
            )
        medians = panel.groupby('outcome_id')['rank_biserial'].median()
        for outcome, value in medians.items():
            if pd.notna(value):
                ax.scatter(value, y_lookup[outcome], marker='D', s=62, color='#111827', zorder=4)
        ax.axvline(0, color='#334155', linewidth=0.9)
        ax.set_xlim(-1.03, 1.03)
        ax.set_title(CONTRAST_LABELS[contrast], fontweight='bold')
        ax.grid(axis='x', color='#E2E8F0', linewidth=0.6)
    axes[0].set_yticks([y_lookup[item] for item in OUTCOME_ORDER], [OUTCOME_LABELS[item] for item in OUTCOME_ORDER], fontsize=12)
    handles = [Line2D([0], [0], marker='o', linestyle='', color=MODEL_COLORS[item], label=MODEL_LABELS[item]) for item in MODEL_ORDER]
    handles += [Line2D([0], [0], marker='o', linestyle='', color='#475569', label='Man'), Line2D([0], [0], marker='^', linestyle='', color='#475569', label='Woman'), Line2D([0], [0], marker='D', linestyle='', color='#111827', label='Median overall')]
    fig.legend(handles=handles, loc='lower center', ncol=5, bbox_to_anchor=(0.5, 0.005))
    fig.supxlabel('Paired rank-biserial persona − pembanding', y=0.065)
    fig.suptitle(f'Overall ETHICS — {title}', fontweight='bold', y=0.995)
    fig.tight_layout(rect=(0.01, 0.12, 0.99, 0.96), w_pad=1.0)
    fig.savefig(FIG_DIR / filename, dpi=180, bbox_inches='tight')
    figure_rows.append((filename, f'Overall {title}.'))
    display(fig)
    plt.close(fig)

plot_overall_within_effects(
    all_text,
    ['explicit_vs_neutral', 'implicit_1_vs_neutral', 'implicit_2_vs_neutral', 'implicit_3_vs_neutral'],
    '02_overall_effects_vs_neutral.png',
    'E, I1, I2, dan I3 terhadap N',
)
plot_overall_within_effects(
    all_text,
    ['implicit_1_vs_explicit', 'implicit_2_vs_explicit', 'implicit_3_vs_explicit'],
    '03_overall_effects_vs_explicit.png',
    'I1, I2, dan I3 terhadap E',
)

# 5.2 Direct Man-Woman visual, retaining the four condition panels.
direct_data = all_text[all_text['analysis_family'].eq('direct_gender')].copy()
fig, axes = plt.subplots(1, 4, figsize=(24, 16), sharex=True, sharey=True)
y_lookup = {outcome: len(OUTCOME_ORDER) - 1 - index for index, outcome in enumerate(OUTCOME_ORDER)}
direct_conditions = ['explicit', 'implicit_1', 'implicit_2', 'implicit_3']
for ax, condition in zip(axes, direct_conditions):
    panel = direct_data[direct_data['contrast_id'].eq(f'man_minus_woman_{condition}')]
    for _, point in panel.iterrows():
        if pd.notna(point['rank_biserial']):
            ax.scatter(point['rank_biserial'], y_lookup[point['outcome_id']], s=58, color=MODEL_COLORS.get(point['model_id'], '#64748B'), alpha=0.95 if point['p_holm'] < 0.05 else 0.45, edgecolors='white', linewidths=0.6)
    medians = panel.groupby('outcome_id')['rank_biserial'].median()
    for outcome, value in medians.items():
        if pd.notna(value):
            ax.scatter(value, y_lookup[outcome], marker='D', s=65, color='#111827', zorder=4)
    ax.axvline(0, color='#334155', linewidth=0.9)
    ax.set_xlim(-1.03, 1.03)
    ax.set_title(condition.replace('_', ' ').title(), fontweight='bold')
    ax.grid(axis='x', color='#E2E8F0', linewidth=0.6)
axes[0].set_yticks([y_lookup[item] for item in OUTCOME_ORDER], [OUTCOME_LABELS[item] for item in OUTCOME_ORDER], fontsize=12)
fig.legend(handles=[Line2D([0], [0], marker='o', linestyle='', color=MODEL_COLORS[item], label=MODEL_LABELS[item]) for item in MODEL_ORDER] + [Line2D([0], [0], marker='D', linestyle='', color='#111827', label='Median overall')], loc='lower center', ncol=4, bbox_to_anchor=(0.5, 0.005))
fig.supxlabel('Paired rank-biserial Man − Woman', y=0.065)
fig.suptitle('Overall ETHICS — perbandingan langsung Man dan Woman', fontweight='bold', y=0.995)
fig.tight_layout(rect=(0.01, 0.12, 0.99, 0.96), w_pad=1.0)
fig.savefig(FIG_DIR / '04_overall_effects_direct_man_woman.png', dpi=180, bbox_inches='tight')
figure_rows.append(('04_overall_effects_direct_man_woman.png', 'Overall direct Man-Woman paired effects.'))
display(fig)
plt.close(fig)

# 5.3 Sensitivity concordance, using the same diagonal reference as r4.
fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharex=True, sharey=True)
for ax, family, title in zip(axes, ['within_identity', 'direct_gender'], ['Within identity', 'Direct Man − Woman']):
    panel = sensitivity[sensitivity['analysis_family'].eq(family)]
    for model_id, points in panel.groupby('model_id'):
        ax.scatter(points['rank_biserial_all'], points['rank_biserial_committed'], s=22, alpha=0.45, color=MODEL_COLORS.get(model_id, '#64748B'), label=MODEL_LABELS.get(model_id, model_id))
    ax.plot((-1, 1), (-1, 1), color='#111827', linewidth=1.0, linestyle='--')
    ax.axhline(0, color='#94A3B8', linewidth=0.7)
    ax.axvline(0, color='#94A3B8', linewidth=0.7)
    ax.set_xlim(-1.03, 1.03)
    ax.set_ylim(-1.03, 1.03)
    ax.set_title(title, fontweight='bold')
    ax.grid(color='#E2E8F0', linewidth=0.5)
axes[0].set_ylabel('Rank-biserial both committed')
fig.supxlabel('Rank-biserial all visible text', y=0.04)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles=handles, labels=labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, 0.08))
fig.suptitle('Overall ETHICS — effect concordance all-visible versus committed', fontweight='bold')
fig.tight_layout(rect=(0.02, 0.16, 0.99, 0.93))
fig.savefig(FIG_DIR / '05_overall_committed_sensitivity_concordance.png', dpi=180, bbox_inches='tight')
figure_rows.append(('05_overall_committed_sensitivity_concordance.png', 'Overall all-visible versus committed effect concordance.'))
display(fig)
plt.close(fig)

# 5.4 Overall opening composition, pooled across models and subsets.
response_path = Path(manifest['source_response_level_markers'])
opening = pd.read_csv(response_path, compression='gzip', usecols=['model_id', 'persona', 'subset', 'opening_category', 'text_available'])
opening['text_available'] = opening['text_available'].astype(str).str.casefold().eq('true')
opening = opening[opening['text_available']].copy()
opening_order = ['deliberative', 'refusal', 'answer_first', 'document_heading', 'step_first', 'bullet_first', 'algebraic', 'instructional', 'problem_restatement', 'contextual', 'narrative']
opening_pct = (
    opening.groupby(['persona', 'opening_category']).size().unstack(fill_value=0)
    .reindex(index=PERSONA_ORDER, columns=opening_order, fill_value=0)
)
opening_pct = opening_pct.div(opening_pct.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(17, 8))
opening_pct.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
ax.set_title('Overall ETHICS — komposisi kategori opening', fontweight='bold')
ax.set_xlabel('Persona')
ax.set_ylabel('Respons linguistically available (%)')
ax.set_ylim(0, 100)
ax.tick_params(axis='x', rotation=35)
ax.legend(title='Opening category', bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
fig.savefig(FIG_DIR / '06_overall_opening_composition_100pct.png', dpi=180, bbox_inches='tight')
figure_rows.append(('06_overall_opening_composition_100pct.png', 'Overall opening-category composition.'))
display(fig)
plt.close(fig)

figure_index_overall = pd.DataFrame(figure_rows, columns=['figure', 'description'])
figure_index_overall['size_bytes'] = figure_index_overall['figure'].map(lambda name: (FIG_DIR / name).stat().st_size)
figure_index_overall.to_csv(OVERALL_DIR / 'figure_index.csv', index=False, encoding='utf-8')
visual_manifest = {
    'result_status': 'candidate',
    'analysis_scope': 'overall_ethics',
    'source_bundle': str(OVERALL_DIR / 'manifest.json'),
    'figures': figure_index_overall.to_dict(orient='records'),
}
(OVERALL_DIR / 'visual_manifest.json').write_text(json.dumps(visual_manifest, indent=2, ensure_ascii=False), encoding='utf-8')
display(figure_index_overall)

## Appendix reference — detail per subset

Detail `commonsense`, `deontology`, `justice`, dan `virtue` tidak dihapus.
Ia tetap tersedia pada notebook lama dan bundle v1.4 r4 berikut. Bagian ini
hanya membuat index lampiran agar hasil utama tetap ringkas.

In [ ]:
appendix_files = [
    'paired_numeric_vs_neutral_all_text.csv',
    'paired_numeric_vs_explicit_all_text.csv',
    'paired_numeric_direct_man_woman_all_text.csv',
    'paired_opening_within_identity_all_text.csv',
    'paired_opening_direct_man_woman_all_text.csv',
]
appendix_index = []
for filename in appendix_files:
    path = APPENDIX_DIR / filename
    table = pd.read_csv(path)
    subset_counts = table.groupby('subset').size().to_dict()
    appendix_index.append({
        'file': filename,
        'rows': len(table),
        'rows_by_subset': '; '.join(f'{key}={value}' for key, value in subset_counts.items()),
        'path': str(path),
    })
display(pd.DataFrame(appendix_index))
print('Notebook detail lama:', REPO_ROOT / 'notebooks/gender/linguistic_marker_gender_ethics_v1_4_1_candidate.ipynb')
print('Bundle detail lama  :', APPENDIX_DIR)
print('Rebuild overall     : python -m src.MAIN_PARSING.ETHICS.build_gender_ethics_linguistic_markers_overall_v1')